---
# STAGE 2: Real Phi-3.5-MoE
### Requirements: 24GB VRAM, ~16GB disk for model download
---

In [ ]:
# Cell 8: Install requirements (run once)
# !pip install transformers>=4.41.0 accelerate datasets peft torch>=2.2.0

# Verify VRAM before proceeding
if DEVICE == 'cuda':
    free_vram = torch.cuda.mem_get_info()[0] / 1e9
    total_vram = torch.cuda.mem_get_info()[1] / 1e9
    print(f"Free VRAM: {free_vram:.1f} GB / {total_vram:.1f} GB")
    if free_vram < 20:
        print("⚠️  WARNING: Less than 20GB free. Clear other GPU processes first.")
        print("   Run: torch.cuda.empty_cache() or restart kernel")
    else:
        print("✅ Sufficient VRAM available")
else:
    print("❌ No CUDA device. Stage 2 requires GPU.")

In [ ]:
!pip install flash_attn

In [1]:
# Cell 9: Load Phi-3.5-mini in bf16 with maximum memory efficiency
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gc

# Clear any lingering VRAM from previous failed runs
torch.cuda.empty_cache()
gc.collect()

MODEL_ID = "microsoft/Phi-tiny-moe-instruct"

print(f"Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model in bf16 with device_map='auto'...")
model_phi = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   
    device_map='auto',            # FIX: Let accelerate distribute memory safely
    low_cpu_mem_usage=True,       # FIX: Force strict memory management during load
    trust_remote_code=False,      
)

# Enable gradient checkpointing to drastically reduce VRAM during backprop
model_phi.gradient_checkpointing_enable()
model_phi.eval()

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"VRAM reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB")

Loading tokenizer from microsoft/Phi-tiny-moe-instruct...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/485 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Model loaded. VRAM used: 3.8 GB
VRAM reserved: 4.0 GB


In [2]:
# Cell 10: Inspect Phi-3.5-MoE architecture to find the right module path
# Run this to understand the exact structure before patching.

print("=" * 60)
print("Model architecture inspection")
print("=" * 60)

# Print top-level structure
print("\nTop-level modules:")
for name, module in model_phi.named_children():
    print(f"  {name}: {type(module).__name__}")

# Find the MoE layers
print("\n\nSearching for MoE layer structure...")
moe_layers_found = []
for name, module in model_phi.named_modules():
    class_name = type(module).__name__
    if 'Expert' in class_name or 'MoE' in class_name or 'expert' in name.lower():
        depth = name.count('.')
        if depth < 5:  # Avoid printing nested noise
            print(f"  [{depth}] {name}: {class_name}")
            moe_layers_found.append((name, module))

# Print one expert's structure in detail
print("\n\nDetailed structure of first expert found:")
if moe_layers_found:
    name, module = moe_layers_found[0]
    print(f"  Path: {name}")
    for pname, param in module.named_parameters():
        print(f"    {pname}: {param.shape} dtype={param.dtype}")

Model architecture inspection

Top-level modules:
  model: PhimoeModel
  lm_head: Linear


Searching for MoE layer structure...
  [4] model.layers.0.mlp.experts: PhimoeExperts
  [4] model.layers.1.mlp.experts: PhimoeExperts
  [4] model.layers.2.mlp.experts: PhimoeExperts
  [4] model.layers.3.mlp.experts: PhimoeExperts
  [4] model.layers.4.mlp.experts: PhimoeExperts
  [4] model.layers.5.mlp.experts: PhimoeExperts
  [4] model.layers.6.mlp.experts: PhimoeExperts
  [4] model.layers.7.mlp.experts: PhimoeExperts
  [4] model.layers.8.mlp.experts: PhimoeExperts
  [4] model.layers.9.mlp.experts: PhimoeExperts
  [4] model.layers.10.mlp.experts: PhimoeExperts
  [4] model.layers.11.mlp.experts: PhimoeExperts
  [4] model.layers.12.mlp.experts: PhimoeExperts
  [4] model.layers.13.mlp.experts: PhimoeExperts
  [4] model.layers.14.mlp.experts: PhimoeExperts
  [4] model.layers.15.mlp.experts: PhimoeExperts
  [4] model.layers.16.mlp.experts: PhimoeExperts
  [4] model.layers.17.mlp.experts: PhimoeExperts


In [3]:
import torch
import torch.nn as nn

class HierarchicalExpert(nn.Module):
    """
    Low-rank LoRA sub-adapter.  Computes: L(x) = (x A^T B^T) * scaling
    
    Key fix: B is zero-initialized by default. This guarantees that at the
    moment of spawn, the sub-adapter's contribution is exactly zero, which
    is what makes Check 2 (zero-loss-spike) provably pass.
    """
    def __init__(self, in_features, out_features, base_rank=16, lora_alpha=32):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.rank    = base_rank
        self.scaling = lora_alpha / base_rank

        # Standard LoRA init: A ~ N(0, 0.01), B = 0
        # B=0 means output is zero at init → no loss spike on insertion
        self.A = nn.Parameter(torch.randn(base_rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, base_rank))   # FIX: was randn

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Cast A and B to match x's dtype at runtime.
        # This handles bf16/fp16/fp32 transparently regardless of how
        # the parameters were initialized.
        A = self.A.to(x.dtype)
        B = self.B.to(x.dtype)
        return (x @ A.t() @ B.t()) * self.scaling

In [4]:
import torch

class SaturationMonitor:
    """
    Bivariate saturation trigger (§3.2 of proposal).

    Fires when BOTH conditions hold over a rolling window of W steps:
      1. Learning Plateau : |d/dt g_k| < tau_plateau
         where g_k = mean(||B[:,r]|| * ||A[r,:]||) is the rank importance score
      2. High Residual    : EMA(L_expert) > alpha * EMA(L_batch)
    """
    def __init__(self, expert_id: int, window: int = 10,
                 tau_plateau: float = 1e-3, alpha: float = 1.05):
        self.expert_id   = expert_id
        self.window      = window
        self.tau_plateau = tau_plateau
        self.alpha       = alpha
        self.ema_decay   = 0.9

        self._ri_history: list[float] = []
        self._ema_expert: float | None = None
        self._ema_batch:  float | None = None

    # ------------------------------------------------------------------
    @staticmethod
    def _rank_importance(A: torch.Tensor, B: torch.Tensor) -> float:
        """
        Per-rank importance: mean_r( ||B[:,r]|| * ||A[r,:]|| )
        Measures how much each rank contributes to the overall output norm.
        """
        with torch.no_grad():
            col_norms = B.detach().float().norm(dim=0)  # [rank]
            row_norms = A.detach().float().norm(dim=1)  # [rank]
            return (col_norms * row_norms).mean().item()

    # ------------------------------------------------------------------
    def update(self,
               lora_A: torch.Tensor, lora_B: torch.Tensor,
               expert_loss: float,   batch_loss: float,
               grad_A=None,          grad_B=None) -> bool:
        """
        Call once per step. Returns True when spawn should be triggered.
        grad_A / grad_B are unused here but kept for API compatibility.
        """
        d = self.ema_decay
        if self._ema_expert is None:
            self._ema_expert = expert_loss
            self._ema_batch  = batch_loss
        else:
            self._ema_expert = d * self._ema_expert + (1 - d) * expert_loss
            self._ema_batch  = d * self._ema_batch  + (1 - d) * batch_loss

        self._ri_history.append(self._rank_importance(lora_A, lora_B))

        if len(self._ri_history) < self.window:
            return False

        # OLS slope of rank-importance over the window
        recent = self._ri_history[-self.window:]
        x = torch.arange(len(recent), dtype=torch.float32)
        y = torch.tensor(recent,      dtype=torch.float32)
        slope = (
            (x * y).mean() - x.mean() * y.mean()
        ) / (x.var(unbiased=False) + 1e-12)

        plateau  = abs(slope.item()) < self.tau_plateau
        high_err = self._ema_expert > self.alpha * self._ema_batch

        return bool(plateau and high_err)

    def reset_after_spawn(self):
        self._ri_history.clear()
        self._ema_expert = None
        self._ema_batch  = None

In [5]:
# Replace SaturationMonitor entirely for calibration.
# Track two domain-separated loss EMAs instead of a fake proxy.

class ConflictSaturationMonitor:
    """
    Fires when BOTH hold for `window` consecutive steps:
      1. Plateau: rank importance slope < tau_plateau  (LoRA stopped learning)
      2. Conflict: |ema_medical - ema_code| > delta_threshold
                   (two domains have diverged in loss — real entanglement signal)
    """
    def __init__(self, tau_plateau=1e-4, delta_threshold=0.5,
                 window=15, beta=0.9):
        self.tau_plateau      = tau_plateau
        self.delta_threshold  = delta_threshold
        self.window           = window
        self.beta             = beta

        self._ri_history: list = []
        self._ema_code    = None
        self._ema_medical = None
        self._plateau_window  = []
        self._conflict_window = []

    def update(self, lora_A, lora_B, loss_val, domain):
        # ── Update domain-separated EMAs ──────────────────────────────────
        b = self.beta
        if domain == "code":
            self._ema_code = (loss_val if self._ema_code is None
                              else b * self._ema_code + (1-b) * loss_val)
        else:
            self._ema_medical = (loss_val if self._ema_medical is None
                                 else b * self._ema_medical + (1-b) * loss_val)

        # ── Rank importance ───────────────────────────────────────────────
        with torch.no_grad():
            col_norms = lora_B.detach().float().norm(dim=0)
            row_norms = lora_A.detach().float().norm(dim=1)
            ri = (col_norms * row_norms).mean().item()
        self._ri_history.append(ri)

        if len(self._ri_history) < self.window:
            return False

        # OLS slope over window
        recent = self._ri_history[-self.window:]
        x = torch.arange(len(recent), dtype=torch.float32)
        y = torch.tensor(recent, dtype=torch.float32)
        slope = ((x*y).mean() - x.mean()*y.mean()) / (x.var(unbiased=False) + 1e-12)
        plateau = abs(slope.item()) < self.tau_plateau

        # Conflict: both domains seen, and their EMAs have diverged
        if self._ema_code is not None and self._ema_medical is not None:
            conflict = abs(self._ema_medical - self._ema_code) > self.delta_threshold
        else:
            conflict = False

        self._plateau_window.append(plateau)
        self._conflict_window.append(conflict)
        if len(self._plateau_window) > self.window:
            self._plateau_window.pop(0)
            self._conflict_window.pop(0)

        if (len(self._plateau_window) == self.window
                and all(self._plateau_window)
                and all(self._conflict_window)):
            self._plateau_window.clear()
            self._conflict_window.clear()
            return True

        return False

    def reset_after_spawn(self):
        self._ri_history.clear()
        self._plateau_window.clear()
        self._conflict_window.clear()

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HierarchicalPhimoeExperts(nn.Module):
    """
    Drop-in replacement for Phi-MoE's packed expert block.

    The SparseMoeBlock calls: self.experts(hidden_states, top_k_ids, top_k_weights)
    where hidden_states is [T, model_dim].  We intercept this, compute the
    original output, and add per-expert LoRA corrections:

        E_k(x) = W_k x + L_{k,0}(x) + Σ_j ReLU(w_{k,j}^T x) L_{k,j}(x)

    NOTE on dimensions
    ------------------
    down_proj.shape = [num_experts, model_dim, ffn_dim]
                                   ^^^^^^^^^  ^^^^^^^^
                                   out_f      in_f (internal ffn dim)

    hidden_states arriving at the experts is [T, model_dim].
    Our LoRA correction therefore lives in model_dim → model_dim space,
    i.e. lora_in = lora_out = out_f = model_dim.
    Using in_f (ffn_dim) here was the critical shape-mismatch bug.
    """

    def __init__(self, original_experts, base_rank: int = 16, lora_alpha: int = 32):
        super().__init__()
        self.original = original_experts

        # Freeze every original expert parameter
        for p in self.original.parameters():
            p.requires_grad = False

        # Infer packed-weight dimensions
        # down_proj: [num_experts, model_dim, ffn_dim]
        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype

        # LoRA correction lives in model_dim space (input to & output from experts)
        # FIX: was using self.in_f (ffn_dim) — wrong dimension for hidden_states
        self.lora_dim = self.out_f   # = model_dim

        # Base LoRA per expert — registered as nn.ModuleList so optimizer finds them
        dev   = self.original.down_proj.device
        dtype = self.original.down_proj.dtype

        self.base_loras = nn.ModuleList([
            HierarchicalExpert(
                in_features=self.lora_dim,
                out_features=self.lora_dim,
                base_rank=base_rank,
                lora_alpha=lora_alpha,
            ).to(dev).to(dtype)
            for _ in range(self.num_experts)
        ])

        # Spawned adapters: plain Python lists, added to optimizer manually on spawn.
        # spawn_loras[k] = [HierarchicalExpert, ...]
        # spawn_gates[k] = [nn.Parameter of shape [lora_dim], ...]
        self.spawn_loras: list[list] = [[] for _ in range(self.num_experts)]
        self.spawn_gates: list[list] = [[] for _ in range(self.num_experts)]

    # ------------------------------------------------------------------
    @property
    def device(self) -> torch.device:
        return self.original.down_proj.device

    # ------------------------------------------------------------------
    def spawn(self, expert_id: int, rank: int = 8,
              weight_grad: torch.Tensor | None = None) -> list:
        """
        Spawn a new ReLU-gated LoRA sub-adapter inside expert `expert_id`.

        Initialization protocol
        -----------------------
        • A  ← top-r rows of V^H from SVD of grad (residual-gradient init, LoRA-GA style)
                Falls back to small random if grad is None or SVD fails.
        • B  ← zero   (guarantees exact zero output at spawn → no loss spike, Check 2)
        • w  ← N(0, σ²) where σ = 1e-3 · Var(W_k)   (per proposal §3.3)

        Returns list of new nn.Parameters to add to the optimizer.
        """
        dev, dtype = self.device, self.dtype

        lora = HierarchicalExpert(
            in_features=self.lora_dim,
            out_features=self.lora_dim,
            base_rank=rank,
            lora_alpha=2 * rank,
        ).to(dev).to(dtype)

        # Gradient-informed A init, B always stays zero
        if weight_grad is not None:
            try:
                # weight_grad: [lora_dim, lora_dim]
                U, S, Vh = torch.linalg.svd(weight_grad.float(), full_matrices=False)
                with torch.no_grad():
                    lora.A.copy_(Vh[:rank].to(dtype))   # top-r rows of V^H
                    # lora.B stays zero → output = 0 at spawn (Check 2 guaranteed)
            except Exception as e:
                print(f"  [spawn] SVD failed ({e}), using default random A init.")

        # Symmetry-breaking gate: σ = c · Var(W_k)  (proposal eq. 3.3)
        # FIX: original used sqrt(var) inconsistently — using var() matches proposal
        sigma = 1e-3 * self.original.down_proj[expert_id].float().var().item()
        gate = nn.Parameter(
            torch.randn(self.lora_dim, device=dev, dtype=dtype) * sigma
        )

        self.spawn_loras[expert_id].append(lora)
        self.spawn_gates[expert_id].append(gate)

        # Return ALL new leaf parameters so the caller can add them to the optimizer
        return list(lora.parameters()) + [gate]

    # ------------------------------------------------------------------
    def forward(self,
                hidden_states: torch.Tensor,   # [T, model_dim]
                top_k_indices: torch.Tensor,   # [T, top_k]
                top_k_weights: torch.Tensor,   # [T, top_k]
                ) -> torch.Tensor:

        orig_out   = self.original.forward(hidden_states, top_k_indices, top_k_weights)
        # FIX: zeros_like inherits device/dtype automatically — avoids multi-GPU mismatch
        correction = torch.zeros_like(orig_out)

        for k in range(self.num_experts):
            mask = (top_k_indices == k)      # [T, top_k]  bool
            if not mask.any():
                continue

            # Effective routing weight for expert k per token (sum over top_k slots)
            # FIX: replaced manual loop with vectorised masked sum
            eff_w = (mask.float() * top_k_weights).sum(dim=1)  # [T]

            # Base LoRA correction
            base_out   = self.base_loras[k](hidden_states)              # [T, lora_dim]
            correction = correction + base_out * eff_w.unsqueeze(-1)

            # Spawned sub-adapter corrections
            for gate_vec, sub_lora in zip(self.spawn_gates[k], self.spawn_loras[k]):
                # ReLU scalar gate per token (proposal §3.1)
                g = F.relu(hidden_states @ gate_vec)                     # [T]
                sub_out    = sub_lora(hidden_states)                     # [T, lora_dim]
                correction = correction + sub_out * (g * eff_w).unsqueeze(-1)

        # Last line of forward(), replacing the current return:
        return (orig_out + correction).to(hidden_states.dtype)

In [7]:
import torch
from accelerate.hooks import remove_hook_from_module, add_hook_to_module

DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_LAYER  = 15
TARGET_EXPERT = 0

# Disable gradient checkpointing — it interacts badly with custom modules
# under accelerate's hook system, and we have so few trainable params
# that VRAM savings are negligible anyway.
model_phi.gradient_checkpointing_disable()

for p in model_phi.parameters():
    p.requires_grad = False

# ── Idempotent unwrap ────────────────────────────────────────────────────────
current = model_phi.model.layers[TARGET_LAYER].mlp.experts
if isinstance(current, HierarchicalPhimoeExperts):
    print("⚠️  Already patched — unwrapping first.")
    original_experts = current.original
    # Restore hook to original before we re-wrap
    if hasattr(current, '_hf_hook'):
        hook = current._hf_hook
        remove_hook_from_module(current)
        add_hook_to_module(original_experts, hook)
    model_phi.model.layers[TARGET_LAYER].mlp.experts = original_experts
else:
    original_experts = current

num_experts, out_f, in_f = original_experts.down_proj.shape
print(f"Layer {TARGET_LAYER} | num_experts={num_experts} | model_dim={out_f} | ffn_dim={in_f}")

# ── Patch ────────────────────────────────────────────────────────────────────
hierarchical_experts = HierarchicalPhimoeExperts(original_experts, base_rank=16, lora_alpha=32)
model_phi.model.layers[TARGET_LAYER].mlp.experts = hierarchical_experts

# ── Transfer the accelerate hook from original → our wrapper ─────────────────
# This is the critical fix: without this, AlignDevicesHook no longer runs on
# this module, so output tensors can drift in device/dtype and corrupt
# hidden_states for all subsequent layers.
if hasattr(original_experts, '_hf_hook'):
    hook = original_experts._hf_hook
    remove_hook_from_module(original_experts)
    add_hook_to_module(hierarchical_experts, hook)
    print("✅ Accelerate hook transferred to wrapper.")
else:
    print("ℹ️  No accelerate hook found on original experts (single-GPU case).")

print("✅ Patch applied.")
trainable = [p for p in hierarchical_experts.parameters() if p.requires_grad]
print(f"Trainable params: {sum(p.numel() for p in trainable):,}")

# ── Smoke-test ───────────────────────────────────────────────────────────────
with torch.no_grad():
    T, top_k = 8, 2
    hs  = torch.randn(T, out_f, device=hierarchical_experts.device, dtype=hierarchical_experts.dtype)
    idx = torch.randint(0, num_experts, (T, top_k), device=hierarchical_experts.device)
    wts = torch.ones(T, top_k, device=hierarchical_experts.device, dtype=hierarchical_experts.dtype) / top_k

    orig_out    = original_experts.forward(hs, idx, wts)
    patched_out = hierarchical_experts.forward(hs, idx, wts)
    diff = (orig_out - patched_out).norm().item()

print(f"Smoke-test ‖orig − patched‖ = {diff:.6f}  (should be ~0.0)")

Layer 15 | num_experts=16 | model_dim=4096 | ffn_dim=448
✅ Accelerate hook transferred to wrapper.
✅ Patch applied.
Trainable params: 2,097,152
Smoke-test ‖orig − patched‖ = 0.000000  (should be ~0.0)


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
model_phi.config.use_cache = False

# ── Dataset ───────────────────────────────────────────────────────────────────
CODE_EXAMPLES = [
    "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    "import numpy as np\nx = np.array([1, 2, 3])\nprint(np.dot(x, x))",
    "class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, x):\n        self.items.append(x)",
    "for i in range(10):\n    if i % 2 == 0:\n        print(f'Even: {i}')",
    "def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[0]",
] * 10

MEDICAL_EXAMPLES = [
    "The patient presented with acute myocardial infarction. ECG showed ST elevation.",
    "Metformin inhibits hepatic gluconeogenesis via AMPK activation.",
    "MRI revealed a 2.3cm hyperintense lesion consistent with glioblastoma.",
    "Sepsis criteria: temperature >38C, heart rate >90bpm, WBC >12000.",
    "The BRCA1 mutation confers a lifetime risk of breast cancer.",
] * 10

all_texts = CODE_EXAMPLES + MEDICAL_EXAMPLES   # code first, then medical (domain shift)

encodings = tokenizer(
    all_texts,
    padding='max_length',
    truncation=True,
    max_length=32,
    return_tensors='pt',
).to(DEVICE)
print(f"Dataset: {len(CODE_EXAMPLES)} code + {len(MEDICAL_EXAMPLES)} medical = "
      f"{len(all_texts)} total  (seq_len=32)")

# ── Monitor & optimiser ───────────────────────────────────────────────────────
monitor = SaturationMonitor(
    expert_id=TARGET_EXPERT,
    window=10,
    tau_plateau=1e-3,
    alpha=1.05,
)

# Only base_loras trainable at start (spawn_loras / spawn_gates added later)
optimizer = torch.optim.AdamW(
    [p for p in hierarchical_experts.parameters() if p.requires_grad],
    lr=2e-5,
)
print(f"Initial trainable params: "
      f"{sum(p.numel() for p in hierarchical_experts.parameters() if p.requires_grad):,}")
print(f"VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB\n")

# ── Helper ────────────────────────────────────────────────────────────────────
def get_base_lora():
    """Convenience accessor for the monitored expert's base LoRA."""
    return hierarchical_experts.base_loras[TARGET_EXPERT]

# ── Training loop ─────────────────────────────────────────────────────────────
N_STEPS    = 100
loss_log   = []
spawn_steps: list[int] = []

# Check flags
check1 = False   # spawn triggered at least once
check2 = None    # |ΔL| < 0.01 immediately after spawn (zero-init guarantee)
check3 = None    # gate gradient is non-zero 2 steps after spawn (dead-zone escape)

print(f"Running {N_STEPS} training steps...")
print("-" * 65)

for step in range(N_STEPS):
    # Sequential sampling enforces code→medical domain shift
    idx       = step % len(all_texts)
    input_ids = encodings['input_ids'][idx].unsqueeze(0)   # [1, 32]

    # Causal LM labels: shift input left by one
    labels          = input_ids.clone()
    labels[:, :-1]  = input_ids[:, 1:]
    labels[:, -1]   = -100   # ignore last position

    # ── Forward + backward ──────────────────────────────────────────────
    optimizer.zero_grad()
    outputs = model_phi(input_ids=input_ids, labels=labels)
    loss    = outputs.loss
    loss_log.append(loss.item())
    loss.backward()

    # ── Feed the saturation monitor ──────────────────────────────────────
    lora = get_base_lora()
    # Proxy expert loss: upweight medical tokens to simulate expert stress
    is_medical        = (idx >= len(CODE_EXAMPLES))
    expert_loss_proxy = loss.item() * (1.15 if is_medical else 0.90)

    triggered = monitor.update(
        lora_A=lora.A.data,
        lora_B=lora.B.data,
        expert_loss=expert_loss_proxy,
        batch_loss=loss.item(),
        grad_A=lora.A.grad,
        grad_B=lora.B.grad,
    )

    # ── Check 1: first spawn event ───────────────────────────────────────
    if triggered and not check1:
        check1 = True
        spawn_steps.append(step)
        print(f"\n✅ CHECK 1 PASS — Spawn triggered at step {step}")

        pre_loss = loss.item()
        print(f"   Loss BEFORE spawn : {pre_loss:.5f}")

        # Build residual gradient for SVD-initialised A  (LoRA-GA §3.3)
        # grad_B: [lora_dim, rank], grad_A: [rank, lora_dim]
        # weight_grad ≈ B^T A  residual approximation: [lora_dim, lora_dim]
        grad_A, grad_B = lora.A.grad, lora.B.grad
        if grad_A is not None and grad_B is not None:
            weight_grad = grad_B.detach().float() @ grad_A.detach().float()
        else:
            weight_grad = None   # spawn() falls back to random A init

        new_params = hierarchical_experts.spawn(
            expert_id=TARGET_EXPERT,
            rank=8,
            weight_grad=weight_grad,
        )
        # Add spawned parameters to the optimiser
        optimizer.add_param_group({'params': new_params, 'lr': 2e-5})
        monitor.reset_after_spawn()

        # ── Check 2: zero-loss spawn ─────────────────────────────────────
        # Because B=0 in the new sub-adapter, its contribution is exactly 0.
        # The loss must therefore not change by more than numerical noise.
        with torch.no_grad():
            post_loss = model_phi(input_ids=input_ids, labels=labels).loss.item()

        delta   = abs(post_loss - pre_loss)
        check2  = delta < 0.01
        print(f"   Loss AFTER spawn  : {post_loss:.5f}")
        print(f"   |Δ Loss|           : {delta:.6f}")
        print(f"{'✅ CHECK 2 PASS' if check2 else '❌ CHECK 2 FAIL'} — "
              f"|Δ| < 0.01  (got {delta:.6f})")

    # ── Optimiser step ───────────────────────────────────────────────────
    optimizer.step()

    # ── Check 3: dead-zone escape ────────────────────────────────────────
    # Why step+2?
    #   step 0 (spawn): B=0, sub_lora output=0, gate gradient=0 (chain rule)
    #   step 1        : optimizer updates B to tiny non-zero; gate can now
    #                   receive gradient because sub_lora output ≠ 0
    #   step 2        : gate.grad is populated with non-zero value → PASS
    if check1 and check3 is None and spawn_steps:
        if step == spawn_steps[0] + 2:
            gates = hierarchical_experts.spawn_gates[TARGET_EXPERT]
            if gates and gates[-1].grad is not None:
                gn     = gates[-1].grad.norm().item()
                check3 = gn > 0
                label  = '✅ CHECK 3 PASS' if check3 else '❌ CHECK 3 FAIL'
                print(f"\n{label} — gate grad_norm = {gn:.3e} at step {step}")
            else:
                print(f"\n⚠️  CHECK 3: gate has no gradient at step {step} "
                      f"(gate list len={len(gates)})")
                check3 = False

    # ── Periodic progress ─────────────────────────────────────────────────
    if step % 20 == 0:
        n_spawned = len(hierarchical_experts.spawn_loras[TARGET_EXPERT])
        vram      = torch.cuda.memory_allocated() / 1e9
        domain    = "medical" if is_medical else "code   "
        print(f"  Step {step:3d} | {domain} | loss={loss.item():.4f} | "
              f"VRAM={vram:.2f} GB | sub_adapters={n_spawned}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STAGE 2 VALIDATION SUMMARY")
print("=" * 65)
print(f"Check 1 (Trigger fires)       : {'✅ PASS' if check1 else '❌ FAIL — increase alpha or reduce tau_plateau'}")
print(f"Check 2 (Zero-loss spawn)     : {'✅ PASS' if check2 else ('❌ FAIL' if check2 is False else '⏭  SKIPPED (Check 1 never fired)')}")
print(f"Check 3 (Dead-zone escape)    : {'✅ PASS' if check3 else ('❌ FAIL' if check3 is False else '⏭  SKIPPED')}")
if spawn_steps:
    print(f"\nSpawn events at steps: {spawn_steps}")
    print(f"Final sub-adapters on expert {TARGET_EXPERT}: "
          f"{len(hierarchical_experts.spawn_loras[TARGET_EXPERT])}")

# STAGE 3
real dataset and eval on single expert

In [8]:
# ── Diagnostics Cell: Jaccard Routing Overlap ────────────────────────────────
# Run BEFORE training to select the best domain conflict pair.
# Uses already-loaded model_phi — no second model load needed.

import torch
from collections import defaultdict
from datasets import load_dataset

# ── Config: edit these to try different pairs ────────────────────────────────
TOP_K          = 2     # Phi-MoE uses top-2 routing (NOT 8 like OLMoE)
N_EXAMPLES     = 60
MAX_LENGTH     = 128   # Keep short to avoid OOM during diagnostic
# Jaccard threshold recalibrated for top-2 routing across 16 experts.
# With top-2, random overlap baseline is ~(2/16)=0.125, so 0.35 is meaningful.
JACCARD_THRESHOLD = 0.35

DATASET_OPTION_A = {
    "name": "Python vs Medical",
    "primary":          ("openai/openai_humaneval", "prompt",        None),
    "conflict":         ("qiaojin/PubMedQA",        "question",      "pqa_labeled"),
}
DATASET_OPTION_B = {
    "name": "Math vs Creative Fiction",
    "primary":          ("DigitalLearningGmbH/MATH-lighteval",           "problem",       "default"),
    "conflict":         ("roneneldan/TinyStories",   "text",          None),
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_text_examples(dataset_name, text_col, config_name=None, n=60, preferred_split="train"):
    """Load examples, automatically falling back to whatever split exists."""
    kwargs = {}
    if config_name:
        kwargs["name"] = config_name

    # Load without specifying split first so we can inspect what's available
    ds_dict = load_dataset(dataset_name, **kwargs)
    
    # Pick split: prefer 'train', then 'test', then whatever exists
    available = list(ds_dict.keys())
    if preferred_split in available:
        split = preferred_split
    elif "test" in available:
        split = "test"
    else:
        split = available[0]
    
    print(f"    (using split='{split}', available={available})")
    ds = ds_dict[split]
    return [str(ex[text_col]) for ex in ds.select(range(min(n, len(ds))))]


def get_top_k_experts_for_domain(model, tokenizer, examples, top_k=2,
                                  n_examples=50, max_length=128):
    """
    Returns normalized frequency dict: (layer_idx, expert_idx) → float.
    Normalized so frequencies sum to 1.0 per domain — comparable across domains.
    """
    expert_counts = defaultdict(int)
    total_activations = 0
    model.eval()

    with torch.no_grad():
        for text in examples[:n_examples]:
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                padding=False,
            ).to(model.device)

            outputs = model(**inputs, output_router_logits=True, return_dict=True)

            if outputs.router_logits is None:
                raise ValueError("router_logits is None")

            for layer_idx, logits in enumerate(outputs.router_logits):
                if logits is None:
                    continue
                if logits.dim() == 3:
                    logits = logits.view(-1, logits.shape[-1])

                top_experts = logits.topk(top_k, dim=-1).indices
                for expert_idx in top_experts.flatten().tolist():
                    expert_counts[(layer_idx, expert_idx)] += 1
                    total_activations += 1

    # Normalize to frequency distribution
    return {k: v / total_activations for k, v in expert_counts.items()}


def compute_weighted_overlap(freq_a: dict, freq_b: dict,
                              concentration_threshold: float = 0.002) -> tuple[float, dict]:
    """
    Only consider experts that are meaningfully concentrated in at least one domain.
    concentration_threshold: minimum frequency in either domain to count the expert.
    
    Returns (jaccard_of_concentrated_experts, per_layer_stats).
    
    With 16 experts and top-2, uniform frequency per layer ≈ 2/16 = 0.125 per layer.
    Across N layers, uniform per (layer, expert) ≈ 0.125/N.
    threshold=0.002 filters out uniformly-distributed experts.
    """
    all_keys = set(freq_a.keys()) | set(freq_b.keys())
    
    # Only keep experts that are non-trivially active in at least one domain
    concentrated = {
        k for k in all_keys
        if freq_a.get(k, 0) >= concentration_threshold
        or freq_b.get(k, 0) >= concentration_threshold
    }
    
    # Of those, find which ones both domains share
    shared = {
        k for k in concentrated
        if freq_a.get(k, 0) >= concentration_threshold
        and freq_b.get(k, 0) >= concentration_threshold
    }
    
    jaccard = len(shared) / len(concentrated) if concentrated else 0.0
    
    # Per-layer breakdown — useful for picking TARGET_LAYER
    layer_stats = defaultdict(lambda: {"shared": 0, "total": 0})
    for (layer_idx, expert_idx) in concentrated:
        layer_stats[layer_idx]["total"] += 1
        if (layer_idx, expert_idx) in shared:
            layer_stats[layer_idx]["shared"] += 1
    
    return jaccard, dict(layer_stats)


# ── Main diagnostic ───────────────────────────────────────────────────────────
def run_jaccard_diagnostic(model, tokenizer, concentration_threshold=0.002):
    print("=" * 65)
    print("JACCARD ROUTING OVERLAP DIAGNOSTIC  (Phi-MoE, top-2)")
    print(f"Threshold: {JACCARD_THRESHOLD}  |  concentration_cutoff: {concentration_threshold}")
    print("=" * 65)

    results     = {}
    layer_stats = {}

    for opt in [DATASET_OPTION_A, DATASET_OPTION_B]:
        name = opt["name"]
        print(f"\n── {name} ──")

        primary_ds,  primary_col,  primary_cfg  = opt["primary"]
        conflict_ds, conflict_col, conflict_cfg  = opt["conflict"]

        print(f"  Loading primary  : {primary_ds}")
        primary_texts  = load_text_examples(primary_ds,  primary_col,  primary_cfg)
        print(f"  Loading conflict : {conflict_ds}")
        conflict_texts = load_text_examples(conflict_ds, conflict_col, conflict_cfg)

        print(f"  Running routing probe...")
        freq_primary  = get_top_k_experts_for_domain(
            model, tokenizer, primary_texts,  top_k=TOP_K, n_examples=N_EXAMPLES)
        freq_conflict = get_top_k_experts_for_domain(
            model, tokenizer, conflict_texts, top_k=TOP_K, n_examples=N_EXAMPLES)

        jaccard, lstats = compute_weighted_overlap(
            freq_primary, freq_conflict, concentration_threshold)
        results[name]     = jaccard
        layer_stats[name] = lstats

        print(f"  Weighted Jaccard : {jaccard:.3f}  "
              f"({'✅ above' if jaccard >= JACCARD_THRESHOLD else '❌ below'} "
              f"threshold {JACCARD_THRESHOLD})")
        
        # Print per-layer overlap — helps choose TARGET_LAYER
        print(f"  Per-layer overlap (shared/total concentrated experts):")
        for layer_idx in sorted(lstats.keys()):
            s = lstats[layer_idx]["shared"]
            t = lstats[layer_idx]["total"]
            bar = "█" * s + "░" * (t - s)
            print(f"    Layer {layer_idx:2d}: {bar}  {s}/{t}")

    # Decision
    print("\n" + "=" * 65)
    print("DECISION")
    print("=" * 65)
    best_name = max(results, key=results.get)
    best_val  = results[best_name]
    for name, j in results.items():
        marker = "◀ SELECTED" if name == best_name else ""
        print(f"  {name:35s}  Jaccard={j:.3f}  {marker}")

    print()
    if best_val >= JACCARD_THRESHOLD:
        print(f"✅ Use: {best_name}")
        # Recommend the layer with highest overlap as TARGET_LAYER
        best_layer = max(
            layer_stats[best_name],
            key=lambda l: layer_stats[best_name][l]["shared"]
        )
        print(f"   Recommended TARGET_LAYER: {best_layer}  "
              f"(most shared expert activations)")
    else:
        best_layer = 15  # fallback
        print(f"⚠️  No pair clears threshold. Using best available: {best_name}")
        print(f"   Falling back to TARGET_LAYER={best_layer}")

    print("=" * 65)
    return results, layer_stats, best_layer


results, layer_stats, recommended_layer = run_jaccard_diagnostic(model_phi, tokenizer)
print(f"\nSet TARGET_LAYER = {recommended_layer} in Cell 11")

JACCARD ROUTING OVERLAP DIAGNOSTIC  (Phi-MoE, top-2)
Threshold: 0.35  |  concentration_cutoff: 0.002

── Python vs Medical ──
  Loading primary  : openai/openai_humaneval
    (using split='test', available=['test'])
  Loading conflict : qiaojin/PubMedQA
    (using split='train', available=['train'])
  Running routing probe...
  Weighted Jaccard : 0.056  (❌ below threshold 0.35)
  Per-layer overlap (shared/total concentrated experts):
    Layer  0: ███░░░░░░░░░░  3/13
    Layer  1: █░░░░░░░░░░░  1/12
    Layer  2: █░░░░░░░░░░░░░  1/14
    Layer  3: ░░░░░░░░░░░░░  0/13
    Layer  4: █░░░░░░░░░  1/10
    Layer  5: ░░░░░░░░░░░  0/11
    Layer  6: ░░░░░░░░  0/8
    Layer  7: █░░░░░░░░  1/9
    Layer  8: ░░░░░░░░  0/8
    Layer  9: ░░░░░░░░░░░  0/11
    Layer 10: ░░░░░░░░░░░  0/11
    Layer 11: ░░░░░░░░░  0/9
    Layer 12: ░░░░░░░░░  0/9
    Layer 13: ░░░░░░░░░  0/9
    Layer 14: █░░░░░░░░  1/9
    Layer 15: █░░░░░░░░  1/9
    Layer 16: ░░░░░░░░░░░  0/11
    Layer 17: █░░░░░░░░░  1/10
    La

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.99M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7500 [00:00<?, ? examples/s]

...

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

    (using split='train', available=['train', 'validation'])
  Running routing probe...
  Weighted Jaccard : 0.069  (❌ below threshold 0.35)
  Per-layer overlap (shared/total concentrated experts):
    Layer  0: ██░░░░░░░  2/9
    Layer  1: ███░░░░░░░░░░  3/13
    Layer  2: █░░░░░░░░░░░░  1/13
    Layer  3: ███░░░░░░░░░░  3/13
    Layer  4: ███░░░░░░░  3/10
    Layer  5: ██░░░░░░░░  2/10
    Layer  6: ░░░░░░░░░░░░  0/12
    Layer  7: █░░░░░░░░░  1/10
    Layer  8: ░░░░░░░░░  0/9
    Layer  9: ░░░░░░░░░  0/9
    Layer 10: ░░░░░░░░░░░░  0/12
    Layer 11: ░░░░░░░░░░░  0/11
    Layer 12: ░░░░░░░░░░  0/10
    Layer 13: ░░░░░░░░  0/8
    Layer 14: █░░░░░░░░░  1/10
    Layer 15: █░░░░░░░░░░░  1/12
    Layer 16: ░░░░░░░░░░░  0/11
    Layer 17: ░░░░░░░░░  0/9
    Layer 18: ░░░░░░  0/6
    Layer 19: ░░░░░░░░░░  0/10
    Layer 20: ░░░░░░░░  0/8
    Layer 21: ░░░░░░░░░  0/9
    Layer 22: ░░░░░░░░░  0/9
    Layer 23: ░░░░░░░░░░  0/10
    Layer 24: █░░░░░░░░  1/9
    Layer 25: ░░░░░░  0/6
    Layer

In [8]:
# ── Shared calibration constants (run this before any calibration cell) ───────
BASE_RANK    = 16
LR           = 2e-5
TARGET_LAYER = 0
TARGET_EXPERT = 0
CALIB_STEPS  = 300

In [9]:
# ── Real Data Loader (adapted from dataset_builder.py, notebook-safe) ────────
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset

# ── Dataset config (inline — no config file needed in notebook) ───────────────
DATASET_CFG = {
    "name":         "Python vs Medical",
    "primary":      ("openai/openai_humaneval", "prompt",   None),
    "conflict":     ("qiaojin/PubMedQA",        "question", "pqa_labeled"),
    "domain_names": ["code", "medical"],
}

# Calibration-specific settings — shorter than full training
MAX_LENGTH_CALIB  = 128    # keep short to fit memory during 9-run grid
N_EACH            = 300    # examples per domain for calibration
CONFLICT_RATIO    = 0.5    # severe conflict — most likely to trigger spawning

def load_split_auto(dataset_name, config_name=None, preferred="train"):
    """Load dataset, auto-selecting split (handles HumanEval test-only etc.)."""
    kwargs = {"name": config_name} if config_name else {}
    ds_dict = load_dataset(dataset_name, **kwargs)
    available = list(ds_dict.keys())
    split = preferred if preferred in available else ("test" if "test" in available else available[0])
    print(f"    [{dataset_name}] using split='{split}' (available: {available})")
    return ds_dict[split]


def build_calibration_dataloader(tokenizer, cfg=DATASET_CFG,
                                  conflict_ratio=CONFLICT_RATIO,
                                  max_length=MAX_LENGTH_CALIB,
                                  n_each=N_EACH):
    """
    Interleave primary + conflict domain examples at given ratio.
    Returns a plain list of tokenized dicts (no DataLoader overhead needed
    for single-sample calibration stepping).
    """
    primary_name, primary_col, primary_cfg = cfg["primary"]
    conflict_name, conflict_col, conflict_cfg = cfg["conflict"]

    print("Loading primary dataset...")
    primary_ds = load_split_auto(primary_name, primary_cfg)
    primary_ds = primary_ds.select(range(min(n_each, len(primary_ds))))

    print("Loading conflict dataset...")
    conflict_ds = load_split_auto(conflict_name, conflict_cfg)
    conflict_ds = conflict_ds.select(range(min(n_each, len(conflict_ds))))

    # Extract text
    primary_texts  = [str(ex[primary_col])  for ex in primary_ds]
    conflict_texts = [str(ex[conflict_col]) for ex in conflict_ds]

    # Interleave at conflict_ratio: for every N conflict examples,
    # include (1-ratio)/ratio primary examples
    n_conflict = int(n_each * conflict_ratio)
    n_primary  = n_each - n_conflict
    primary_texts  = primary_texts[:n_primary]
    conflict_texts = conflict_texts[:n_conflict]

    # Build interleaved list: alternate to create domain shift pattern
    import math
    texts, domains = [], []
    ratio = n_conflict / max(n_primary, 1)
    ci = pi = 0
    conflict_acc = 0.0
    while pi < len(primary_texts) or ci < len(conflict_texts):
        # Add primary
        if pi < len(primary_texts):
            texts.append(primary_texts[pi]); domains.append("code"); pi += 1
        # Add conflict proportionally
        conflict_acc += ratio
        while conflict_acc >= 1.0 and ci < len(conflict_texts):
            texts.append(conflict_texts[ci]); domains.append("medical"); ci += 1
            conflict_acc -= 1.0

    print(f"\nDataset built: {len(texts)} total "
          f"({domains.count('code')} code / {domains.count('medical')} medical)")

    # Tokenize
    encodings = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )

    return encodings, domains   # return both so we know domain at each step


# Build once, reuse across all 9 calibration runs
print("Building calibration dataset...")
calib_encodings, calib_domains = build_calibration_dataloader(tokenizer)
print("Done.")

Building calibration dataset...
Loading primary dataset...


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

    [openai/openai_humaneval] using split='test' (available: ['test'])
Loading conflict dataset...


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

    [qiaojin/PubMedQA] using split='train' (available: ['train'])

Dataset built: 300 total (150 code / 150 medical)
Done.


In [11]:
# ── Trigger Diagnostic: why is nothing firing? ────────────────────────────────
# Run ONE trial manually and log the two trigger signals every step
# so we can see whether plateau, high_error, or both are failing.

import torch
gc.collect(); torch.cuda.empty_cache()

# Fresh patch
current = model_phi.model.layers[TARGET_LAYER].mlp.experts
if isinstance(current, HierarchicalPhimoeExperts):
    original_experts = current.original
    if hasattr(current, '_hf_hook'):
        h = current._hf_hook
        remove_hook_from_module(current); add_hook_to_module(original_experts, h)
    model_phi.model.layers[TARGET_LAYER].mlp.experts = original_experts
else:
    original_experts = current

hier = HierarchicalPhimoeExperts(original_experts, base_rank=BASE_RANK, lora_alpha=BASE_RANK*2)
model_phi.model.layers[TARGET_LAYER].mlp.experts = hier
if hasattr(original_experts, '_hf_hook'):
    h = original_experts._hf_hook
    remove_hook_from_module(original_experts); add_hook_to_module(hier, h)

optimizer = torch.optim.AdamW(
    [p for p in hier.parameters() if p.requires_grad], lr=LR)

# Use aggressive settings to maximise chance of firing
test_alpha       = 1.05
test_tau_plateau = 1e-4
monitor = SaturationMonitor(expert_id=TARGET_EXPERT, window=10,
                             tau_plateau=test_tau_plateau, alpha=test_alpha)

print(f"Diagnostic run: alpha={test_alpha}, tau_plateau={test_tau_plateau}")
print(f"{'Step':>5} {'domain':>8} {'loss':>8} {'exp_proxy':>10} "
      f"{'ri_slope':>12} {'plateau':>8} {'high_err':>9}")
print("-" * 70)

ri_history = []
ema_decay  = 0.9
ema_expert = None
ema_batch  = None

for step in range(100):
    idx       = step % len(calib_domains)
    input_ids = calib_encodings['input_ids'][idx].unsqueeze(0)
    labels    = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:]
    labels[:, -1]  = -100

    optimizer.zero_grad()
    loss = model_phi(input_ids=input_ids, labels=labels).loss
    loss.backward()

    lora = hier.base_loras[TARGET_EXPERT]
    is_medical = (calib_domains[idx] == "medical")
    expert_proxy = loss.item() * (1.15 if is_medical else 0.90)

    # Manually compute what the monitor sees
    if lora.A.grad is not None and lora.B.grad is not None:
        # Rank importance: mean(||B[:,r]|| * ||A[r,:]||)
        col_norms = lora.B.data.detach().float().norm(dim=0)
        row_norms = lora.A.data.detach().float().norm(dim=1)
        ri = (col_norms * row_norms).mean().item()
    else:
        ri = 0.0
    ri_history.append(ri)

    # OLS slope over last 10 steps
    if len(ri_history) >= 10:
        recent = ri_history[-10:]
        x = torch.arange(10, dtype=torch.float32)
        y = torch.tensor(recent, dtype=torch.float32)
        slope = ((x*y).mean() - x.mean()*y.mean()) / (x.var(unbiased=False) + 1e-12)
        slope_val = slope.item()
    else:
        slope_val = float('nan')

    # EMA tracking
    d = ema_decay
    ema_expert = expert_proxy if ema_expert is None else d*ema_expert + (1-d)*expert_proxy
    ema_batch  = loss.item()  if ema_batch  is None else d*ema_batch  + (1-d)*loss.item()

    plateau   = abs(slope_val) < test_tau_plateau if not torch.isnan(torch.tensor(slope_val)) else False
    high_err  = ema_expert > test_alpha * ema_batch

    if step % 5 == 0:
        domain_str = "medical" if is_medical else "code"
        print(f"{step:>5} {domain_str:>8} {loss.item():>8.4f} {expert_proxy:>10.4f} "
              f"{slope_val:>12.2e} {str(plateau):>8} {str(high_err):>9}")

    optimizer.step()

print("\n── Key values at final step ──")
print(f"  ema_expert = {ema_expert:.4f}")
print(f"  ema_batch  = {ema_batch:.4f}")
print(f"  ratio      = {ema_expert/ema_batch:.4f}  (need > {test_alpha})")
print(f"  ri range   = [{min(ri_history):.2e}, {max(ri_history):.2e}]")
print(f"  slope range ≈ last value = {slope_val:.2e}  (need < {test_tau_plateau})")

Diagnostic run: alpha=1.05, tau_plateau=0.0001
 Step   domain     loss  exp_proxy     ri_slope  plateau  high_err
----------------------------------------------------------------------
    0     code  12.9739    11.6765          nan    False     False


KeyboardInterrupt: 

In [ ]:
# ── Phase 2: Hyperparameter Calibration Grid ─────────────────────────────────
# Goal: find (alpha, tau_plateau) that gives 1-3 spawns per 100 steps.
# 9 combinations × 300 steps each. Uses existing HierarchicalPhimoeExperts
# and SaturationMonitor — no architecture changes from Stage 2.

import gc, torch, itertools
from accelerate.hooks import remove_hook_from_module, add_hook_to_module

calib_results = []   # will hold dicts with settings + spawn count

# ── Updated calibration grid ──────────────────────────────────────────────────
DELTA_VALUES      = [0.3, 0.5, 1.0]    # |ema_medical - ema_code| threshold
TAU_PLATEAU_VALUES = [1e-3, 5e-4, 1e-4]

def run_calibration_trial_v2(delta_threshold, tau_plateau,
                              encodings, domains, n_steps=CALIB_STEPS):
    gc.collect(); torch.cuda.empty_cache()

    # Fresh patch (same unwrap logic as before)
    current = model_phi.model.layers[TARGET_LAYER].mlp.experts
    if isinstance(current, HierarchicalPhimoeExperts):
        original_experts = current.original
        if hasattr(current, '_hf_hook'):
            h = current._hf_hook
            remove_hook_from_module(current); add_hook_to_module(original_experts, h)
        model_phi.model.layers[TARGET_LAYER].mlp.experts = original_experts
    else:
        original_experts = current

    hier = HierarchicalPhimoeExperts(original_experts, base_rank=BASE_RANK,
                                      lora_alpha=BASE_RANK*2)
    model_phi.model.layers[TARGET_LAYER].mlp.experts = hier
    if hasattr(original_experts, '_hf_hook'):
        h = original_experts._hf_hook
        remove_hook_from_module(original_experts); add_hook_to_module(hier, h)

    monitor = ConflictSaturationMonitor(
        tau_plateau=tau_plateau,
        delta_threshold=delta_threshold,
        window=15,
    )
    optimizer = torch.optim.AdamW(
        [p for p in hier.parameters() if p.requires_grad], lr=LR)

    spawn_steps, loss_log = [], []

    for step in range(n_steps):
        idx       = step % len(domains)
        input_ids = encodings['input_ids'][idx].unsqueeze(0)
        labels    = input_ids.clone()
        labels[:, :-1] = input_ids[:, 1:]
        labels[:, -1]  = -100

        optimizer.zero_grad()
        loss = model_phi(input_ids=input_ids, labels=labels).loss
        loss_log.append(loss.item())
        loss.backward()

        lora   = hier.base_loras[TARGET_EXPERT]
        domain = domains[idx]

        triggered = monitor.update(
            lora_A=lora.A.data,
            lora_B=lora.B.data,
            loss_val=loss.item(),
            domain=domain,
        )

        if triggered:
            grad_A, grad_B = lora.A.grad, lora.B.grad
            weight_grad = (grad_B.detach().float() @ grad_A.detach().float()
                           if grad_A is not None and grad_B is not None else None)
            new_params = hier.spawn(TARGET_EXPERT, rank=8, weight_grad=weight_grad)
            optimizer.add_param_group({'params': new_params, 'lr': LR})
            monitor.reset_after_spawn()
            spawn_steps.append(step)

        optimizer.step()

    spawn_rate = len(spawn_steps) / (n_steps / 100)
    return {
        "delta_threshold": delta_threshold,
        "tau_plateau":     tau_plateau,
        "spawns":          len(spawn_steps),
        "spawn_steps":     spawn_steps,
        "spawn_rate":      spawn_rate,
        "final_loss":      loss_log[-1],
    }


# Run updated grid
print("=" * 65)
print("PHASE 2 (v2): CALIBRATION WITH CONFLICT DELTA TRIGGER")
print(f"Grid: delta={DELTA_VALUES} × tau_plateau={TAU_PLATEAU_VALUES}")
print("=" * 65)

calib_results_v2 = []
for delta, tau in itertools.product(DELTA_VALUES, TAU_PLATEAU_VALUES):
    print(f"\n── delta={delta}  tau_plateau={tau:.0e} ──")
    r = run_calibration_trial_v2(delta, tau, calib_encodings, calib_domains)
    calib_results_v2.append(r)
    tag = ("✅ GOOD" if 1 <= r["spawn_rate"] <= 3
           else "🔥 TOO MANY" if r["spawn_rate"] > 3 else "❌ NONE")
    print(f"   Spawns: {r['spawns']}  |  Rate: {r['spawn_rate']:.1f}/100 steps  |  {tag}")

print("\n" + "=" * 65)
print("SUMMARY")
print(f"{'delta':>8}  {'tau_plateau':>12}  {'spawns':>7}  {'rate/100':>9}  verdict")
print("-" * 65)
good = []
for r in calib_results_v2:
    rate = r["spawn_rate"]
    tag  = "✅ GOOD" if 1 <= rate <= 3 else ("🔥 TOO MANY" if rate > 3 else "❌ NONE")
    print(f"{r['delta_threshold']:>8.1f}  {r['tau_plateau']:>12.0e}  "
          f"{r['spawns']:>7}  {rate:>9.1f}  {tag}")
    if 1 <= rate <= 3:
        good.append(r)

print("=" * 65)
if good:
    best = min(good, key=lambda r: r["spawn_rate"])
    print(f"\n✅ LOCKED HYPERPARAMETERS:")
    print(f"   delta_threshold = {best['delta_threshold']}")
    print(f"   tau_plateau     = {best['tau_plateau']:.0e}")
    LOCKED_DELTA       = best["delta_threshold"]
    LOCKED_TAU_PLATEAU = best["tau_plateau"]
else:
    print("\n⚠️  Still no spawns. Check that loss variance between domains is >0.3")
    print("   Print monitor._ema_code and monitor._ema_medical mid-run to verify.")

In [10]:
# ── Phase 1 / Phase 4: JSD Sub-Router Specialisation Probe ───────────────────
# Measures Jensen-Shannon Divergence between ReLU gate activation distributions
# P_code and P_medical on each spawned sub-adapter.
#
# High JSD (→1.0) = gates have partitioned the two domains → entanglement resolved
# Low JSD  (→0.0) = gates treat both domains identically → no specialisation
#
# Call compute_jsd_probe() at checkpoints during Phase 3 training.

import torch
import torch.nn.functional as F
import numpy as np
from collections import defaultdict

def collect_gate_activations(hier_experts, tokenizer, texts, domain_label,
                              max_length=128, n_examples=50):
    activations = defaultdict(list)
    model_phi.eval()

    with torch.no_grad():
        for text in texts[:n_examples]:
            inputs = tokenizer(
                text, return_tensors='pt',
                truncation=True, max_length=max_length, padding=False
            ).to(hier_experts.device)

            captured = {}
            def hook_fn(module, input, output):
                h = input[0].detach()
                # FIX: flatten batch/seq dimensions → always [T, model_dim]
                if h.dim() == 3:
                    h = h.view(-1, h.shape[-1])
                captured['hidden'] = h

            hook = model_phi.model.layers[TARGET_LAYER].mlp.register_forward_hook(hook_fn)
            _ = model_phi(**inputs)
            hook.remove()

            if 'hidden' not in captured:
                continue

            hidden = captured['hidden']  # now guaranteed [T, model_dim]

            for j, gate_vec in enumerate(hier_experts.spawn_gates[TARGET_EXPERT]):
                gate_acts = F.relu(hidden @ gate_vec.to(hidden.dtype))  # [T]
                activations[j].extend(gate_acts.float().cpu().tolist())

    model_phi.train()
    return activations


def compute_jsd(p_vals, q_vals, n_bins=50):
    """
    Compute Jensen-Shannon Divergence between two empirical distributions.
    Bins both into a shared histogram, then computes JSD.
    Returns scalar in [0, 1] (using log base 2, so max=1).
    """
    if len(p_vals) < 5 or len(q_vals) < 5:
        return float('nan')

    # Shared bin range
    all_vals = p_vals + q_vals
    min_v, max_v = min(all_vals), max(all_vals)
    if max_v - min_v < 1e-8:
        return 0.0   # both distributions are identical point masses

    bins = np.linspace(min_v, max_v, n_bins + 1)
    p_hist, _ = np.histogram(p_vals, bins=bins, density=False)
    q_hist, _ = np.histogram(q_vals, bins=bins, density=False)

    # Normalise to probability distributions with Laplace smoothing
    p = (p_hist + 1e-8) / (p_hist.sum() + 1e-8 * n_bins)
    q = (q_hist + 1e-8) / (q_hist.sum() + 1e-8 * n_bins)

    m = 0.5 * (p + q)
    # JSD = 0.5 * KL(P||M) + 0.5 * KL(Q||M), base-2
    jsd = 0.5 * np.sum(p * np.log2(p / m + 1e-12)) + \
          0.5 * np.sum(q * np.log2(q / m + 1e-12))
    return float(np.clip(jsd, 0.0, 1.0))


def compute_jsd_probe(hier_experts, tokenizer,
                       code_texts, medical_texts,
                       step=None, n_examples=50, verbose=True):
    """
    Main probe function. Call this at training checkpoints.

    Returns dict: {sub_adapter_idx: jsd_value}
    A JSD > 0.3 is meaningful specialisation.
    A JSD > 0.6 is strong evidence of domain partitioning.
    """
    n_spawned = len(hier_experts.spawn_gates[TARGET_EXPERT])
    if n_spawned == 0:
        if verbose:
            print("  [JSD probe] No sub-adapters spawned yet — skipping.")
        return {}

    if verbose:
        label = f" (step {step})" if step is not None else ""
        print(f"\n── JSD Probe{label} ──  "
              f"{n_spawned} sub-adapter(s) on L{TARGET_LAYER} E{TARGET_EXPERT}")

    code_acts    = collect_gate_activations(
        hier_experts, tokenizer, code_texts,    "code",    n_examples=n_examples)
    medical_acts = collect_gate_activations(
        hier_experts, tokenizer, medical_texts, "medical", n_examples=n_examples)

    jsd_results = {}
    for j in range(n_spawned):
        p_code    = code_acts.get(j, [])
        p_medical = medical_acts.get(j, [])
        jsd       = compute_jsd(p_code, p_medical)
        jsd_results[j] = jsd

        if verbose:
            bar_len = int(jsd * 20) if not np.isnan(jsd) else 0
            bar     = "█" * bar_len + "░" * (20 - bar_len)
            verdict = ("🟢 strong" if jsd > 0.6
                       else "🟡 moderate" if jsd > 0.3
                       else "🔴 weak" if not np.isnan(jsd)
                       else "⚪ n/a")
            print(f"  Sub-adapter {j}: JSD={jsd:.3f}  [{bar}]  {verdict}")

    return jsd_results


# ── Build held-out probe texts (no overlap with training data) ────────────────
# Use the LAST n examples from each dataset — training uses the first N_EACH
# ── Build held-out probe texts ────────────────────────────────────────────────
print("Building held-out probe texts...")

PROBE_N = 50

primary_name,  primary_col,  primary_cfg  = DATASET_CFG["primary"]
conflict_name, conflict_col, conflict_cfg = DATASET_CFG["conflict"]

_primary_full  = load_split_auto(primary_name,  primary_cfg)
_conflict_full = load_split_auto(conflict_name, conflict_cfg)

# HumanEval only has 164 examples and was never used for training
# (training used calibration encodings, not this dataset directly),
# so the full split is clean to use as probe.
probe_code_texts = [
    str(ex[primary_col])
    for ex in _primary_full.select(range(min(PROBE_N, len(_primary_full))))
]

# PubMedQA has 1000 examples — skip first N_EACH to avoid train overlap
_conflict_skip = min(N_EACH, len(_conflict_full) - PROBE_N)
probe_medical_texts = [
    str(ex[conflict_col])
    for ex in _conflict_full.select(
        range(_conflict_skip, min(_conflict_skip + PROBE_N, len(_conflict_full)))
    )
]

print(f"Probe texts: {len(probe_code_texts)} code, "
      f"{len(probe_medical_texts)} medical (held-out)")

# FIX: use whatever the current patched experts variable is called.
# Grab it directly from the model so this cell works regardless of
# what the calibration cell named its local variable.
# Ensure we're probing a freshly patched, untrained wrapper
# (calibration left a trained hier in the layer — reset it)
current = model_phi.model.layers[TARGET_LAYER].mlp.experts
if isinstance(current, HierarchicalPhimoeExperts):
    original_experts = current.original
    if hasattr(current, '_hf_hook'):
        h = current._hf_hook
        remove_hook_from_module(current)
        add_hook_to_module(original_experts, h)
    model_phi.model.layers[TARGET_LAYER].mlp.experts = original_experts

# Fresh patch for probing
original_experts = model_phi.model.layers[TARGET_LAYER].mlp.experts
hier_experts = HierarchicalPhimoeExperts(
    original_experts, base_rank=BASE_RANK, lora_alpha=BASE_RANK*2)
model_phi.model.layers[TARGET_LAYER].mlp.experts = hier_experts
if hasattr(original_experts, '_hf_hook'):
    h = original_experts._hf_hook
    remove_hook_from_module(original_experts)
    add_hook_to_module(hier_experts, h)

print(f"Fresh patch applied. Sub-adapters: "
      f"{len(hier_experts.spawn_gates[TARGET_EXPERT])}")  # should be 0

print("\nTest run on current model state:")
_ = compute_jsd_probe(hier_experts, tokenizer,
                       probe_code_texts, probe_medical_texts, step=0)

Building held-out probe texts...
    [openai/openai_humaneval] using split='test' (available: ['test'])
    [qiaojin/PubMedQA] using split='train' (available: ['train'])
Probe texts: 50 code, 50 medical (held-out)
Fresh patch applied. Sub-adapters: 0

Test run on current model state:
  [JSD probe] No sub-adapters spawned yet — skipping.


In [11]:
# ── Phase 3 Config ─────────────────────────────────────────────────────────────
# All settings in one place. Do not change after grid starts.

PHASE3_CFG = {
    # Locked from Phase 2
    "delta_threshold": 1.0,
    "tau_plateau":     1e-4,
    "target_layer":    0,
    "target_expert":   0,
    "base_rank":       16,
    "lr":              2e-5,
    "sigma_scale":     1e-3,

    # Training
    "n_steps":         1500,     # per run — enough to see negative transfer emerge
    "max_sub_adapters": 10,     # cap on spawns per expert
    "eval_every":      100,     # steps between JSD + perplexity checkpoints
    "max_length":      128,
    "batch_size":      1,

    # Grid
    "conflict_ratios": {
        "A": 0.0,   # 100% code
        "B": 0.2,   # 80/20
        "C": 0.5,   # 50/50
    },
    "methods": ["lora", "dr_lora", "hierarchical"],
}

# Evaluation: held-out perplexity as proxy for Python accuracy
# (avoids needing code execution infrastructure)
# Lower perplexity on code probe = less negative transfer
EVAL_CODE_TEXTS    = probe_code_texts      # from JSD probe cell
EVAL_MEDICAL_TEXTS = probe_medical_texts

print("Phase 3 config locked.")
print(f"Total runs: {len(PHASE3_CFG['methods'])} methods × "
      f"{len(PHASE3_CFG['conflict_ratios'])} ratios = "
      f"{len(PHASE3_CFG['methods']) * len(PHASE3_CFG['conflict_ratios'])} training runs")
print(f"Steps per run: {PHASE3_CFG['n_steps']}")

Phase 3 config locked.
Total runs: 3 methods × 3 ratios = 9 training runs
Steps per run: 1500


In [12]:
# ── Method Factory ─────────────────────────────────────────────────────────────
# Returns (patched_model_component, optimizer, monitor_or_none) for each method.
# Critically: always starts from a clean unpatched layer.

import torch.nn as nn
from accelerate.hooks import remove_hook_from_module, add_hook_to_module

def get_clean_original_experts(target_layer):
    """
    Always returns raw original experts, unwrapping any wrapper type.
    Handles HierarchicalPhimoeExperts, DRLoRAExperts, or any future wrapper
    that stores the original as .original.
    """
    current = model_phi.model.layers[target_layer].mlp.experts

    # Keep unwrapping until we hit the raw packed experts
    # (which have .down_proj as a raw tensor, not an nn.Module attribute)
    while hasattr(current, 'original'):
        original = current.original
        if hasattr(current, '_hf_hook'):
            h = current._hf_hook
            remove_hook_from_module(current)
            add_hook_to_module(original, h)
        model_phi.model.layers[target_layer].mlp.experts = original
        current = original

    return current


class DRLoRAExperts(nn.Module):
    """
    DR-LoRA baseline: per-expert LoRA that doubles rank at a fixed midpoint step.
    
    This is a deliberate simplification: the paper's saliency-based trigger
    is designed for single-domain resource allocation, not cross-domain conflict.
    A fixed-step expansion gives a clean controlled comparison — same capacity
    budget and timing control as hierarchical, but no independent gating.
    The entanglement hypothesis predicts this will underperform hierarchical
    under conflict regardless of trigger timing.
    """
    def __init__(self, original_experts, base_rank=16, lora_alpha=32,
                 expand_at_step=750):
        super().__init__()
        self.original      = original_experts
        self.expand_at_step = expand_at_step
        self.expanded       = False

        for p in self.original.parameters():
            p.requires_grad = False

        _, self.out_f, self.in_f = self.original.down_proj.shape
        self.num_experts = self.original.down_proj.shape[0]
        self.lora_dim    = self.out_f
        self.dtype       = self.original.down_proj.dtype

        dev = self.original.down_proj.device

        # Per-expert base LoRA (rank 8, grows to 16 at expansion)
        # Using rank 8 init so total budget after expansion = rank 16 = same as LoRA baseline
        self.base_loras = nn.ModuleList([
            HierarchicalExpert(
                in_features=self.lora_dim,
                out_features=self.lora_dim,
                base_rank=base_rank // 2,   # starts at 8
                lora_alpha=lora_alpha // 2,
            ).to(dev).to(self.dtype)
            for _ in range(self.num_experts)
        ])

        # Expansion LoRAs — allocated at init but zero-contribution until expanded
        self.exp_loras = nn.ModuleList([
            HierarchicalExpert(
                in_features=self.lora_dim,
                out_features=self.lora_dim,
                base_rank=base_rank // 2,
                lora_alpha=lora_alpha // 2,
            ).to(dev).to(self.dtype)
            for _ in range(self.num_experts)
        ])
        # Disable exp_loras from optimizer until expansion
        for lora in self.exp_loras:
            for p in lora.parameters():
                p.requires_grad = False

        self._rank_growth_events = []

    @property
    def device(self):
        return self.original.down_proj.device

    def maybe_expand_rank(self, step, optimizer):
        if self.expanded or step < self.expand_at_step:
            return
        # Unlock expansion LoRAs and add to optimizer
        new_params = []
        for lora in self.exp_loras:
            for p in lora.parameters():
                p.requires_grad = True
                new_params.append(p)
        optimizer.add_param_group({'params': new_params, 'lr': PHASE3_CFG["lr"]})
        self.expanded = True
        self._rank_growth_events.append(step)
        print(f"    [DR-LoRA] Rank expanded at step {step}")

    def forward(self, hidden_states, top_k_indices, top_k_weights):
        orig_out   = self.original.forward(hidden_states, top_k_indices, top_k_weights)
        correction = torch.zeros_like(orig_out)

        for k in range(self.num_experts):
            mask = (top_k_indices == k)
            if not mask.any():
                continue
            eff_w = (mask.float() * top_k_weights).sum(dim=1)
            correction = correction + self.base_loras[k](hidden_states) * eff_w.unsqueeze(-1)
            if self.expanded:
                correction = correction + self.exp_loras[k](hidden_states) * eff_w.unsqueeze(-1)

        return (orig_out + correction).to(hidden_states.dtype)


def setup_method(method_name, cfg):
    """
    Patch the model for the given method. Returns (experts_wrapper, optimizer).
    Call get_clean_original_experts first to ensure clean state.
    """
    target_layer = cfg["target_layer"]
    original     = get_clean_original_experts(target_layer)
    model_phi.gradient_checkpointing_disable()
    for p in model_phi.parameters():
        p.requires_grad = False

    if method_name == "lora":
        # Standard LoRA: HierarchicalPhimoeExperts with no spawning allowed
        wrapper = HierarchicalPhimoeExperts(
            original, base_rank=cfg["base_rank"], lora_alpha=cfg["base_rank"]*2)
        model_phi.model.layers[target_layer].mlp.experts = wrapper
        _transfer_hook(original, wrapper)
        optimizer = torch.optim.AdamW(
            [p for p in wrapper.parameters() if p.requires_grad], lr=cfg["lr"])
        return wrapper, optimizer, None   # no monitor

    elif method_name == "dr_lora":
        # DR-LoRA: shared rank-growth LoRA
        # growth_threshold: rolling loss level that triggers expansion
        wrapper = DRLoRAExperts(
            original,
            base_rank=cfg["base_rank"],
            lora_alpha=cfg["base_rank"] * 2,
            expand_at_step=750,
        )
        model_phi.model.layers[target_layer].mlp.experts = wrapper
        _transfer_hook(original, wrapper)
        optimizer = torch.optim.AdamW(
            [p for p in wrapper.parameters() if p.requires_grad], lr=cfg["lr"])
        return wrapper, optimizer, None

    elif method_name == "hierarchical":
        wrapper = HierarchicalPhimoeExperts(
            original, base_rank=cfg["base_rank"], lora_alpha=cfg["base_rank"]*2)
        model_phi.model.layers[target_layer].mlp.experts = wrapper
        _transfer_hook(original, wrapper)
        monitor = ConflictSaturationMonitor(
            tau_plateau=cfg["tau_plateau"],
            delta_threshold=cfg["delta_threshold"],
            window=15,
        )
        optimizer = torch.optim.AdamW(
            [p for p in wrapper.parameters() if p.requires_grad], lr=cfg["lr"])
        return wrapper, optimizer, monitor

    else:
        raise ValueError(f"Unknown method: {method_name}")


def _transfer_hook(src, dst):
    """Transfer accelerate hook from src to dst if present."""
    if hasattr(src, '_hf_hook'):
        h = src._hf_hook
        remove_hook_from_module(src)
        add_hook_to_module(dst, h)

print("Method factory ready.")

Method factory ready.


In [13]:
# ── Evaluation Helper ──────────────────────────────────────────────────────────
# Measures perplexity on held-out code and medical texts.
# Perplexity increase on code (relative to Run A ceiling) = negative transfer.

def evaluate_perplexity(texts, tokenizer, max_length=128, n_eval=30):
    """
    Returns mean perplexity over texts. Lower = better retention of domain.
    """
    model_phi.eval()
    losses = []
    with torch.no_grad():
        for text in texts[:n_eval]:
            inputs = tokenizer(
                text, return_tensors='pt', truncation=True,
                max_length=max_length, padding=False
            ).to(model_phi.device)
            if inputs['input_ids'].shape[1] < 2:
                continue
            labels          = inputs['input_ids'].clone()
            labels[:, :-1]  = inputs['input_ids'][:, 1:]
            labels[:, -1]   = -100
            loss = model_phi(**inputs, labels=labels).loss
            losses.append(loss.item())
    model_phi.train()
    if not losses:
        return float('nan')
    mean_loss = sum(losses) / len(losses)
    return float(torch.exp(torch.tensor(mean_loss)).item())  # perplexity


def evaluate_checkpoint(wrapper, method_name, run_label, step, cfg):
    """
    Run full evaluation at a checkpoint. Returns dict of metrics.
    """
    metrics = {"method": method_name, "run": run_label, "step": step}

    # Perplexity on both domains
    metrics["ppl_code"]    = evaluate_perplexity(
        EVAL_CODE_TEXTS, tokenizer, cfg["max_length"])
    metrics["ppl_medical"] = evaluate_perplexity(
        EVAL_MEDICAL_TEXTS, tokenizer, cfg["max_length"])

    # JSD (only meaningful if sub-adapters exist)
    jsd_scores = {}
    if method_name == "hierarchical":
        jsd_scores = compute_jsd_probe(
            wrapper, tokenizer,
            EVAL_CODE_TEXTS, EVAL_MEDICAL_TEXTS,
            step=step, verbose=False
        )
    metrics["jsd_scores"] = jsd_scores
    metrics["mean_jsd"]   = (sum(jsd_scores.values()) / len(jsd_scores)
                              if jsd_scores else float('nan'))

    n_spawns = (len(wrapper.spawn_gates[cfg["target_expert"]])
                if method_name == "hierarchical" else
                (1 if (method_name == "dr_lora" and wrapper.expanded) else 0))
    metrics["n_expansions"] = n_spawns

    print(f"  [{method_name} | Run {run_label} | step {step}]  "
          f"ppl_code={metrics['ppl_code']:.2f}  "
          f"ppl_medical={metrics['ppl_medical']:.2f}  "
          f"mean_jsd={metrics['mean_jsd']:.3f}  "
          f"expansions={n_spawns}")
    return metrics

print("Evaluation helper ready.")

Evaluation helper ready.


In [14]:
# ── Phase 3 Grid Runner ────────────────────────────────────────────────────────
import gc, itertools

all_results  = []   # flat list of checkpoint metric dicts
all_run_logs = {}   # (method, run_label) -> loss_log

model_phi.config.use_cache = False

for method_name, (run_label, conflict_ratio) in itertools.product(
        PHASE3_CFG["methods"], PHASE3_CFG["conflict_ratios"].items()):

    run_key = (method_name, run_label)
    print("\n" + "=" * 65)
    print(f"METHOD: {method_name.upper()}  |  Run {run_label} "
          f"(conflict={conflict_ratio:.0%})")
    print("=" * 65)

    gc.collect(); torch.cuda.empty_cache()

    # ── Build dataset for this conflict ratio ─────────────────────────────────
    enc, domains = build_calibration_dataloader(
        tokenizer,
        conflict_ratio=conflict_ratio,
        max_length=PHASE3_CFG["max_length"],
        n_each=PHASE3_CFG["n_steps"] + 50,   # enough for full run
    )

    # ── Setup method ──────────────────────────────────────────────────────────
    wrapper, optimizer, monitor = setup_method(method_name, PHASE3_CFG)

    loss_log   = []
    spawn_log  = []

    # ── Training loop ─────────────────────────────────────────────────────────
    for step in range(PHASE3_CFG["n_steps"]):
        idx       = step % len(domains)
        input_ids = enc['input_ids'][idx].unsqueeze(0)
        labels    = input_ids.clone()
        labels[:, :-1] = input_ids[:, 1:]
        labels[:, -1]  = -100

        optimizer.zero_grad()
        loss = model_phi(input_ids=input_ids, labels=labels).loss
        loss_log.append(loss.item())
        loss.backward()

        # ── Method-specific post-backward logic ───────────────────────────────
        if method_name == "dr_lora":
           wrapper.maybe_expand_rank(step, optimizer)

        elif method_name == "hierarchical":
            lora   = wrapper.base_loras[PHASE3_CFG["target_expert"]]
            domain = domains[idx]

            if lora.A.grad is not None and lora.B.grad is not None:
                triggered = monitor.update(
                    lora_A=lora.A.data, lora_B=lora.B.data,
                    loss_val=loss.item(), domain=domain,
                )
                if (triggered and
                        len(wrapper.spawn_gates[PHASE3_CFG["target_expert"]])
                        < PHASE3_CFG["max_sub_adapters"]):
                    grad_A, grad_B = lora.A.grad, lora.B.grad
                    wg = (grad_B.detach().float() @ grad_A.detach().float()
                          if grad_A is not None else None)
                    new_params = wrapper.spawn(
                        PHASE3_CFG["target_expert"], rank=8, weight_grad=wg)
                    optimizer.add_param_group(
                        {'params': new_params, 'lr': PHASE3_CFG["lr"]})
                    monitor.reset_after_spawn()
                    spawn_log.append(step)

        optimizer.step()

        # ── Checkpoint evaluation ─────────────────────────────────────────────
        if (step + 1) % PHASE3_CFG["eval_every"] == 0:
            metrics = evaluate_checkpoint(
                wrapper, method_name, run_label, step + 1, PHASE3_CFG)
            metrics["loss_at_step"] = loss.item()
            all_results.append(metrics)

    all_run_logs[run_key] = loss_log
    if spawn_log:
        print(f"  Spawn events: {spawn_log}")

# ── Results summary ───────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("PHASE 3 COMPLETE — NEGATIVE TRANSFER SUMMARY")
print("=" * 65)

# Get Run A ceiling perplexity for each method (baseline = no conflict)
print(f"\n{'Method':>14} {'Run':>4} {'PPL_code':>10} {'PPL_med':>9} "
      f"{'NegTransfer':>13} {'MeanJSD':>9}")
print("-" * 65)

# Ceiling = last checkpoint of Run A for each method
ceilings = {}
for method_name in PHASE3_CFG["methods"]:
    run_a = [r for r in all_results
             if r["method"] == method_name and r["run"] == "A"]
    if run_a:
        ceilings[method_name] = run_a[-1]["ppl_code"]

for r in all_results:
    # Only print final checkpoint per run
    method, run = r["method"], r["run"]
    if r["step"] != PHASE3_CFG["n_steps"]:
        continue
    ceil = ceilings.get(method, float('nan'))
    neg_transfer = r["ppl_code"] - ceil if not (
        torch.isnan(torch.tensor(ceil))) else float('nan')
    jsd_str = f"{r['mean_jsd']:.3f}" if not torch.isnan(
        torch.tensor(r['mean_jsd'])) else "  n/a"
    print(f"{method:>14} {run:>4} {r['ppl_code']:>10.2f} "
          f"{r['ppl_medical']:>9.2f} {neg_transfer:>+13.2f} {jsd_str:>9}")

print("=" * 65)
print("\nPositive NegTransfer = perplexity increased = Python got worse.")
print("Hierarchical should show lowest NegTransfer on Runs B and C.")


METHOD: LORA  |  Run A (conflict=0%)
Loading primary dataset...
    [openai/openai_humaneval] using split='test' (available: ['test'])
Loading conflict dataset...
    [qiaojin/PubMedQA] using split='train' (available: ['train'])

Dataset built: 164 total (164 code / 0 medical)
  [lora | Run A | step 100]  ppl_code=148755.31  ppl_medical=1808988.62  mean_jsd=nan  expansions=0
  [lora | Run A | step 200]  ppl_code=102242.41  ppl_medical=1796342.50  mean_jsd=nan  expansions=0
  [lora | Run A | step 300]  ppl_code=13853.21  ppl_medical=1651952.75  mean_jsd=nan  expansions=0
  [lora | Run A | step 400]  ppl_code=754.81  ppl_medical=1589590.38  mean_jsd=nan  expansions=0
  [lora | Run A | step 500]  ppl_code=390.83  ppl_medical=1617450.88  mean_jsd=nan  expansions=0
  [lora | Run A | step 600]  ppl_code=287.04  ppl_medical=1631931.75  mean_jsd=nan  expansions=0
  [lora | Run A | step 700]  ppl_code=189.52  ppl_medical=1602364.88  mean_jsd=nan  expansions=0
  [lora | Run A | step 800]  ppl_c

In [15]:
# ── Eager Spawn Ablation: Hierarchical Run C with lower delta_threshold ────
# Tests whether earlier spawn timing recovers Run C performance.
# All settings identical to Phase 3 except delta_threshold=0.5 and max_sub_adapters=15.

EAGER_CFG = {**PHASE3_CFG, "delta_threshold": 0.5, "max_sub_adapters": 15}

gc.collect(); torch.cuda.empty_cache()

enc, domains = build_calibration_dataloader(
    tokenizer,
    conflict_ratio=0.5,
    max_length=PHASE3_CFG["max_length"],
    n_each=PHASE3_CFG["n_steps"] + 50,
)

wrapper, optimizer, monitor = setup_method("hierarchical", EAGER_CFG)

eager_results = []
spawn_log = []

for step in range(EAGER_CFG["n_steps"]):
    idx       = step % len(domains)
    input_ids = enc['input_ids'][idx].unsqueeze(0)
    labels    = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:]
    labels[:, -1]  = -100

    optimizer.zero_grad()
    loss = model_phi(input_ids=input_ids, labels=labels).loss
    loss.backward()

    lora   = wrapper.base_loras[EAGER_CFG["target_expert"]]
    domain = domains[idx]

    if lora.A.grad is not None and lora.B.grad is not None:
        triggered = monitor.update(
            lora_A=lora.A.data, lora_B=lora.B.data,
            loss_val=loss.item(), domain=domain,
        )
        if (triggered and
                len(wrapper.spawn_gates[EAGER_CFG["target_expert"]])
                < EAGER_CFG["max_sub_adapters"]):
            grad_A, grad_B = lora.A.grad, lora.B.grad
            wg = (grad_B.detach().float() @ grad_A.detach().float()
                  if grad_A is not None else None)
            new_params = wrapper.spawn(
                EAGER_CFG["target_expert"], rank=8, weight_grad=wg)
            optimizer.add_param_group(
                {'params': new_params, 'lr': EAGER_CFG["lr"]})
            monitor.reset_after_spawn()
            spawn_log.append(step)

    optimizer.step()

    if (step + 1) % EAGER_CFG["eval_every"] == 0:
        metrics = evaluate_checkpoint(
            wrapper, "hierarchical_eager", "C", step + 1, EAGER_CFG)
        eager_results.append(metrics)

print(f"\nEager spawn events: {spawn_log}")
print(f"First spawn at step: {spawn_log[0] if spawn_log else 'none'}")
print(f"Compare to standard Run C first spawn: 234")
print(f"Final PPL: {eager_results[-1]['ppl_code']:.2f}")
print(f"Compare to standard Run C final PPL:  579.59")

Loading primary dataset...
    [openai/openai_humaneval] using split='test' (available: ['test'])
Loading conflict dataset...
    [qiaojin/PubMedQA] using split='train' (available: ['train'])

Dataset built: 939 total (164 code / 775 medical)
  [hierarchical_eager | Run C | step 100]  ppl_code=121089.20  ppl_medical=1775257.12  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 200]  ppl_code=84595.59  ppl_medical=1730614.25  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 300]  ppl_code=8773.00  ppl_medical=1728794.75  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 400]  ppl_code=18539.57  ppl_medical=1723640.38  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 500]  ppl_code=18885.44  ppl_medical=1737440.75  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 600]  ppl_code=18647.75  ppl_medical=1719667.00  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 700]  ppl_code=18493.05  ppl_medical=17374